# Chapter 1 — Hello Transformers

This notebook is an annotated walkthrough of the first chapter of **_Natural Language Processing with Transformers_**: getting hands-on with Transformer models before going deep into tokenizers, model internals, fine-tuning, or deployment.

The chapter’s central idea is simple but important: with Hugging Face `pipeline()`, you can run many common NLP tasks using pretrained models with only a few lines of code. In this notebook, one customer-support complaint is reused across several tasks so the differences between tasks become easy to compare:

- **Text classification** estimates the sentiment of the message.
- **Named entity recognition** finds entities such as organizations, places, characters, and names.
- **Question answering** extracts the answer span directly from the original text.
- **Summarization** compresses the complaint into its main point.
- **Translation** converts the English text into German.
- **Text generation** asks GPT-2 to continue a customer-service response.

A useful mental model for every pipeline in this notebook is:

```text
raw text → tokenizer/preprocessing → pretrained Transformer model → task-specific postprocessing → readable output
```

The notebook is intentionally output-preserving: the original execution results are kept inside the notebook so you can study the tables, warnings, downloaded model logs, generated text, and translation even before rerunning anything.


## 1. Imports

We begin with the minimal libraries needed for this chapter. `pandas` is only used to display dictionaries returned by pipelines as readable tables. `transformers.pipeline` is the high-level API that hides tokenization, model loading, inference, and postprocessing behind one simple interface.


In [3]:
# pandas is used only for clean tabular display of pipeline outputs.
import pandas as pd

# Importing transformers directly is useful when checking package behavior/version.
import transformers

# pipeline() is the high-level Hugging Face API used throughout this chapter.
from transformers import pipeline

## 2. Shared input text

The same fictional complaint is used throughout the notebook. This is a good teaching pattern because each cell changes the **task**, not the input, so the output differences come from the model/pipeline behavior rather than from new examples.


In [4]:
# A fictional customer-support complaint used as the single running example.
# It contains sentiment, named entities, a clear request, and enough context for
# summarization, translation, question answering, and text generation.
text = """Dear Amazon, last week I ordered an Optimus Prime action figure
from your online store in Germany. Unfortunately, when I opened the package,
I discovered to my horror that I had been sent an action figure of Megatron
instead! As a lifelong enemy of the Decepticons, I hope you can understand my
dilemma. To resolve the issue, I demand an exchange of Megatron for the
Optimus Prime figure I ordered. Enclosed are copies of my records concerning
this purchase. I expect to hear from you soon. Sincerely, Bumblebee."""

## 3. Build a text-classification pipeline

The first pipeline performs sentiment classification. Since no model name is passed here, Transformers selects its default model for the task and prints a warning. For learning notebooks this is acceptable, but for serious projects it is better to specify the exact model and revision for reproducibility.


In [5]:
# Create a pretrained sentiment-analysis/text-classification pipeline.
# Because no model is specified, Transformers loads its default model for this task.
classifier = pipeline('text-classification')

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Device set to use cpu


## 4. Run sentiment analysis

Now the classifier is applied to the customer complaint. The pipeline returns a list of dictionaries with a predicted label and confidence score. Converting the result to a `pandas.DataFrame` makes the output easier to inspect.


In [7]:
# Run the classifier on the customer complaint.
cls_outputs = classifier(text)

# Display the returned list of dictionaries as a clean table.
pd.DataFrame(cls_outputs)

,label,score
0,NEGATIVE,0.901546


## 5. Named entity recognition

Named entity recognition, or NER, identifies spans of text that refer to entities such as organizations, people, locations, and miscellaneous named objects. The `aggregation_strategy="simple"` argument groups subword-level predictions into cleaner entity chunks, which is more readable than raw token-level output.


In [8]:
# Create a named entity recognition pipeline.
# aggregation_strategy="simple" merges token/subword predictions into readable spans.
ner_tagger = pipeline("ner", aggregation_strategy="simple")

# Extract entities from the same complaint text.
ner_outputs = ner_tagger(text)

# Show entity type, confidence score, text span, and character offsets.
pd.DataFrame(ner_outputs)

No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496 (https://huggingface.co/dbmdz/bert-large-cased-finetuned-conll03-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Device set to use cpu


,entity_group,score,word,start,end
0,ORG,0.879010,Amazon,5,11
1,MISC,0.990859,Optimus Prime,36,49
2,LOC,0.999755,Germany,90,97
3,MISC,0.556570,Mega,208,212
4,PER,0.590256,##tron,212,216
5,ORG,0.669692,Decept,253,259
6,MISC,0.498349,##icons,259,264
7,MISC,0.775362,Megatron,350,358
8,MISC,0.987854,Optimus Prime,367,380
9,PER,0.812096,Bumblebee,502,511


## 6. Extractive question answering

Question answering here is **extractive**: the model receives a question and a context paragraph, then returns the most likely answer span copied from the context. This is different from generative QA, where a model may produce an answer in its own words.


In [11]:
# Create an extractive question-answering pipeline.
reader = pipeline("question-answering")

# Ask a question whose answer is present inside the complaint.
question = "What does the customer want?"

# The model searches the context and returns the most likely answer span.
qa_outputs = reader(question=question, context=text)

# Wrap the dictionary in a list so pandas displays it as one table row.
pd.DataFrame([qa_outputs])

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5 (https://huggingface.co/distilbert/distilbert-base-cased-distilled-squad).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu


,score,start,end,answer
0,0.631292,335,358,an exchange of Megatron


## 7. Summarization

Summarization is a sequence-to-sequence task: the model reads the full complaint and generates a shorter version. The `max_length` parameter limits the summary length, and `clean_up_tokenization_spaces=True` makes the final text cleaner.


In [15]:
# Create a summarization pipeline.
summarizer = pipeline("summarization")

# Generate a short summary of the complaint.
# max_length limits the generated summary length.
sum_outputs = summarizer(text, max_length=45, clean_up_tokenization_spaces=True)

# Print only the generated summary text from the pipeline output dictionary.
print(sum_outputs[0]['summary_text'])

No model was supplied, defaulted to sshleifer/distilbart-cnn-12-6 and revision a4f8f3e (https://huggingface.co/sshleifer/distilbart-cnn-12-6).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu
Your min_length=56 must be inferior than your max_length=45.


 Bumblebee ordered an Optimus Prime action figure from your online store in Germany. Unfortunately, when I opened the package, I discovered to my horror that I had been sent an action figure of Megatron instead.


## 8. Translation pipeline

This cell creates an English-to-German translation pipeline. The original notebook used this model family from Helsinki-NLP; in the annotated version the model identifier is written as `Helsinki-NLP/opus-mt-en-de` so the notebook is safer to rerun.


In [18]:
# Create an English-to-German translation pipeline.
# Note: the model namespace is Helsinki-NLP.
translator = pipeline("translation_en_to_de", model="Helsinki-NLP/opus-mt-en-de")

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/298M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/298M [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/768k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/797k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Device set to use cpu


## 9. Run translation

The translation pipeline returns a list containing a dictionary with the generated German text. Because the input is fairly long, `min_length=100` encourages the model to produce a sufficiently detailed translation instead of stopping too early.


In [23]:
# Translate the full complaint into German.
# min_length encourages the model to produce a sufficiently detailed translation.
tr_outputs = translator(text, clean_up_tokenization_spaces=True, min_length=100)

# Print the translated text returned by the pipeline.
print(tr_outputs[0]['translation_text'])

Sehr geehrter Amazon, letzte Woche habe ich eine Optimus Prime Action Figur aus Ihrem Online-Shop in Deutschland bestellt. Leider, als ich das Paket öffnete, entdeckte ich zu meinem Entsetzen, dass ich stattdessen eine Action Figur von Megatron geschickt worden war! Als lebenslanger Feind der Decepticons, Ich hoffe, Sie können mein Dilemma verstehen. Um das Problem zu lösen, Ich fordere einen Austausch von Megatron für die Optimus Prime Figur habe ich bestellt. Eingeschlossen sind Kopien meiner Aufzeichnungen über diesen Kauf. Ich erwarte, von Ihnen bald zu hören. Aufrichtig, Bumblebee.


## 10. Build a text-generation pipeline

The final task uses a causal language model for open-ended text generation. As with the first classifier cell, not passing an explicit model makes Transformers load its default GPT-2 model for this pipeline.


In [24]:
# Create a text-generation pipeline.
# With no explicit model, Transformers loads the default GPT-2 model.
generator = pipeline("text-generation")

No model was supplied, defaulted to openai-community/gpt2 and revision 607a30d (https://huggingface.co/openai-community/gpt2).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


## 11. Generate a customer-service response

Here the original complaint is combined with the beginning of a support response. The generator then continues the text. Unlike classification, generation can vary across runs because decoding depends on model version, random sampling settings, and generation parameters.


In [27]:
# Start the customer-service response with a polite opening.
response = "Dear Bumblebee, I am sorry to hear that order was mixed up."

# Combine the original complaint and the partial response into one generation prompt.
prompt = text + "

Customer Service response:
" + response

# Ask the model to continue the prompt.
gen_outputs = generator(prompt, max_length=200)

# Print the full generated sequence: original prompt plus continuation.
print(gen_outputs[0]['generated_text'])

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Dear Amazon, last week I ordered an Optimus Prime action figure
from your online store in Germany. Unfortunately, when I opened the package,
I discovered to my horror that I had been sent an action figure of Megatron
instead! As a lifelong enemy of the Decepticons, I hope you can understand my
dilemma. To resolve the issue, I demand an exchange of Megatron for the
Optimus Prime figure I ordered. Enclosed are copies of my records concerning
this purchase. I expect to hear from you soon. Sincerely, Bumblebee.

Customer Service response:
Dear Bumblebee, I am sorry to hear that order was mixed up. I was not able to contact you directly and

thank you for your help, but I have to say that I wish to apologize to my customers. I have been

very disappointed using your website and your online store to return my order. We are very

in disbelief that you would refuse to return this product and would not take any action to

return it. If to your knowledge, this could not be the case, but I can te

## 12. End of chapter checkpoint

At this point, the notebook has shown the same input text moving through several common NLP tasks. The purpose is not to train models yet; it is to build intuition for what pretrained Transformer pipelines can do and what their outputs look like.


In [ ]:
# Empty checkpoint cell kept from the original notebook structure.
# You can use this cell for experiments after completing the chapter.
